# Multispectral Ocean Debris Segmentation Engine
## 14-Channel UNet Training & Ablation Pipeline (MARIDA)

This notebook trains and evaluates the deep learning model described in `plan.md` Section 7 and `execution_plan.md` Phase 1.

- **Inputs**: 14 channels (11 Sentinel-2 L2A bands + FDI, NDVI, NDWI)
- **Loss**: Combined Soft-Dice Loss + Focal Loss
- **Ablation Study**:
  - Model A: RGB (B4, B3, B2)
  - Model B: 11 Sentinel-2 bands
  - Model C: 11 bands + FDI + NDVI + NDWI (14 channels)
- **Export**: ONNX with numerical parity verification ($< 10^{-4}$)

In [ ]:
# Environment Setup
!pip install -q onnx onnxruntime rasterio pyproj shapely scipy

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')

### 1. Data Loader & Spectral Index Computation

In [ ]:
def compute_fdi(b8, b6, b11):
    return b8 - (b6 + (b11 - b6) * 1.772)

def compute_normalized_ratio(a, b, eps=1e-6):
    return np.clip((a - b) / (a + b + eps), -1.0, 1.0)

def compute_ndvi(b8, b4):
    return compute_normalized_ratio(b8, b4)

def compute_ndwi(b3, b8):
    return compute_normalized_ratio(b3, b8)

### 2. UNet Architecture Definition

In [ ]:
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.net(x)

class DebrisUNet(nn.Module):
    def __init__(self, in_channels=14, base=32):
        super().__init__()
        self.inc = DoubleConv(in_channels, base)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(base, base*2))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(base*2, base*4))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(base*4, base*8))
        
        self.up1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv_up1 = DoubleConv(base*8 + base*4, base*4)
        self.up2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv_up2 = DoubleConv(base*4 + base*2, base*2)
        self.up3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv_up3 = DoubleConv(base*2 + base, base)
        self.outc = nn.Conv2d(base, 1, 1)
        
    def forward(self, x, return_prob=True):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        
        x = self.conv_up1(torch.cat([x3, self.up1(x4)], dim=1))
        x = self.conv_up2(torch.cat([x2, self.up2(x)], dim=1))
        x = self.conv_up3(torch.cat([x1, self.up3(x)], dim=1))
        logits = self.outc(x)
        return torch.sigmoid(logits) if return_prob else logits

print('UNet defined successfully.')

### 3. Loss Functions & Training Loop

In [ ]:
class DiceFocalLoss(nn.Module):
    def __init__(self, alpha=0.8, gamma=2.0, eps=1e-6):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.eps = eps
        
    def forward(self, probs, targets):
        # Soft Dice
        p = probs.view(-1)
        t = targets.view(-1)
        intersection = (p * t).sum()
        dice = (2.0 * intersection + self.eps) / (p.sum() + t.sum() + self.eps)
        dice_loss = 1.0 - dice
        
        # Focal
        p_clamp = torch.clamp(p, self.eps, 1.0 - self.eps)
        pt = torch.where(t == 1, p_clamp, 1.0 - p_clamp)
        alpha_t = torch.where(t == 1, self.alpha, 1.0 - self.alpha)
        focal_loss = (-alpha_t * ((1.0 - pt) ** self.gamma) * torch.log(pt)).mean()
        
        return 0.5 * dice_loss + 0.5 * focal_loss

### 4. ONNX Export and Numerical Parity Check

In [ ]:
import onnx
import onnxruntime as ort

def export_and_verify(model, output_path='debris_unet.onnx'):
    model.eval()
    dummy = torch.randn(1, 14, 256, 256, device=device)
    
    torch.onnx.export(
        model,
        dummy,
        output_path,
        input_names=['input'],
        output_names=['output'],
        opset_version=14,
        dynamic_axes={
            'input': {0: 'batch_size', 2: 'height', 3: 'width'},
            'output': {0: 'batch_size', 2: 'height', 3: 'width'}
        }
    )
    
    sess = ort.InferenceSession(output_path, providers=['CPUExecutionProvider'])
    ort_out = sess.run(['output'], {'input': dummy.cpu().numpy()})[0]
    with torch.no_grad():
        torch_out = model(dummy, return_prob=True).cpu().numpy()
        
    diff = float(np.max(np.abs(torch_out - ort_out)))
    print(f'Max absolute difference between PyTorch and ONNX: {diff:.6e}')
    assert diff < 1e-4, 'Parity check failed!'
    print('SUCCESS: Verified ONNX model export.')